# Ragas 0.4.3 예제 모음

`RAGAS_0.4_GUIDE.md` 의 내용을 실제로 실행해 보는 노트북입니다.

| 예제 | 내용 | 가이드 |
|---|---|---|
| 1 | 지식 그래프를 **직접** 만들고 저장하기 (한 번 만들어 재사용) | 2-4 |
| 2 | 페르소나 + 한국어 프롬프트 + 질문 유형 비율로 테스트셋 생성 | 2-5, 2-6 |
| 3 | 평가 대상 RAG 만들고 답변 수집 | 3-1 |
| 4 | `metrics.collections` (새 방식)으로 채점 + 채점 근거 보기 | 3-2 B, 3-3 |
| 5 | 설정값 비교: retriever `k` 에 따라 검색 지표가 어떻게 바뀌나 | 3-3 |

> 비용을 줄이기 위해 PDF 5페이지, 질문 4개로 작게 실행합니다. (gpt-4o-mini 기준 수십 원 수준)

In [19]:
from dotenv import load_dotenv

load_dotenv()  # OPENAI_API_KEY

False

## 준비: 문서 로드

In [20]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()[3:8]  # 본문 5페이지만 사용 (비용 절약)

print(f"페이지 수: {len(docs)}")
print(docs[0].page_content[:300])

페이지 수: 5
1. 정책/법제  
2. 기업/산업 
3. 기술/연구 
 4. 인력/교육
미국, 안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령 발표 
n 미국 바이든 대통령이 ‘안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령’에 서명하고 
광범위한 행정 조치를 명시
n 행정명령은 △AI의 안전과 보안 기준 마련 △개인정보보호 △형평성과 시민권 향상 △소비자 
보호 △노동자 지원 △혁신과 경쟁 촉진 △국제협력을 골자로 함
KEY Contents
£ 바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 


## 준비: 테스트셋 생성용 LLM / 임베딩

테스트셋 생성(`ragas.testset`)은 LangChain 모델을 **래퍼로 감싸서** 사용합니다.
(`TestsetGenerator.from_langchain()` 이 내부에서 하는 일과 같습니다.)

In [21]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import RunConfig
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
generator_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# 무료/저등급 API 키에서도 429(rate limit) 에러가 나지 않도록 동시 요청 수를 낮춥니다.
run_config = RunConfig(max_workers=4, timeout=120, max_retries=3)

C:\Users\user\AppData\Local\Temp\ipykernel_3912\2901548992.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
C:\Users\user\AppData\Local\Temp\ipykernel_3912\2901548992.py:7: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


---
## 예제 1. 지식 그래프 직접 만들기

`generate_with_langchain_docs()` 는 호출할 때마다 **요약·개체명 추출을 처음부터 다시** 합니다.
그래프를 한 번 만들어 저장해 두면, 질문 유형이나 개수를 바꿔 가며 여러 번 생성해도 추출 비용이 들지 않습니다.

```
Document ─▶ Node(DOCUMENT) ─▶ default_transforms ─▶ 요약/제목/분할/개체명/주제 + 관계
```

In [22]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

kg = KnowledgeGraph()
for doc in docs:
    # 각 페이지를 DOCUMENT 노드로 추가 (generate_with_langchain_docs 가 내부에서 하는 일)
    kg.add(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata},
        )
    )
kg

KnowledgeGraph(nodes: 5, relationships: 0)

`default_transforms()` 는 문서 길이 분포를 보고 파이프라인을 **자동으로** 고릅니다.
어떤 단계들이 선택됐는지 출력해 봅니다.

In [23]:
from ragas.testset.transforms import apply_transforms, default_transforms

transforms = default_transforms(documents=docs, llm=generator_llm, embedding_model=generator_emb)

for t in transforms:
    # Parallel 은 내부에 여러 단계를 묶고 있음
    inner = getattr(t, "transformations", None)
    print(type(t).__name__, [type(x).__name__ for x in inner] if inner else "")

HeadlinesExtractor 
HeadlineSplitter 
SummaryExtractor 
CustomNodeFilter 
Parallel ['EmbeddingExtractor', 'ThemesExtractor', 'NERExtractor']
Parallel ['CosineSimilarityBuilder', 'OverlapScoreBuilder']


In [24]:
# 그래프에 변환 적용 (LLM 호출이 가장 많은 단계)
apply_transforms(kg, transforms, run_config=run_config)

# 노드 종류별 개수 / 관계 수 확인
from collections import Counter

print(Counter(n.type.name for n in kg.nodes))
print("관계 수:", len(kg.relationships), Counter(r.type for r in kg.relationships))

Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 1001.98it/s]

Counter({'CHUNK': 10, 'DOCUMENT': 5})
관계 수: 26 Counter({'entities_overlap': 11, 'child': 10, 'next': 5})


In [25]:
# 청크 노드 하나가 어떤 속성을 갖게 됐는지 확인
chunk = next(n for n in kg.nodes if n.type == NodeType.CHUNK)
for k, v in chunk.properties.items():
    if k.endswith("embedding"):
        v = f"<벡터 {len(v)}차원>"
    print(f"{k:20} {str(v)[:120]}")

page_content         KEY Contents £ 바이든 대통령, AI 행정명령 통해 안전하고 신뢰할 수 있는 AI 개발과 활용 추진 n 미국 바이든 대통령이 2023년 10월 30일 연방정부 차원에서 안전하고 신뢰할 수 있는 AI 개발과
themes               ['AI development and utilization', 'Biden administration', 'Safety and security standards for AI', 'Data privacy', 'Equi
entities             ['바이든 대통령', 'AI', '미국', '2023년 10월 30일', 'FLOPS', '국가AI연구자원', 'NAIRR']


In [26]:
# 저장해 두면 다음부터 KnowledgeGraph.load(...) 로 재사용 가능 (추출 비용 0)
kg.save("../data/ragas_kg.json")
# kg = KnowledgeGraph.load("../data/ragas_kg.json")

---
## 예제 2. 페르소나 + 한국어 + 질문 유형 비율

- **페르소나**: 자동 생성(영어) 대신 실제 사용자층을 직접 지정
- **한국어**: synthesizer 프롬프트의 예시를 `adapt_prompts("korean")` 으로 번역
- **질문 유형**: 그래프에 재료(관계)가 있는 synthesizer 만 골라 같은 비율로

> 💡 예제 1 결과에서 관계 종류를 보세요. `summary_similarity` 관계가 없으면(문서가 적거나 주제가 서로 달라 요약 유사도가 0.7 미만)
> **MultiHopAbstract** 질문은 만들 수 없어 자동으로 제외됩니다. 문서를 늘리면 생깁니다.

In [27]:
from ragas.testset import TestsetGenerator
from ragas.testset.persona import Persona

# ⚠️ name 은 영어로! 한글 이름을 쓰면 내부 매칭 LLM 이 이름을 영어로 바꿔 답해서
#    KeyError: "No persona found with name 'Policy Researcher'" 가 발생합니다.
#    설명(role_description)은 한국어로 써도 됩니다.
personas = [
    Persona(name="Policy Researcher", role_description="각국 정부의 AI 규제와 국제 협력 동향을 비교·분석한다."),
    Persona(name="Startup CTO", role_description="생성 AI 모델 도입, 비용, 오픈소스 여부에 관심이 많다."),
]

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_emb,
    knowledge_graph=kg,      # 예제 1 에서 만든 그래프 재사용
    persona_list=personas,
)

In [28]:
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
)

# llm_context: 질문 생성 방향 힌트. ragas 는 질문 스타일(문법 오류·검색어 형태 등)을 일부러 섞는데,
# 이때 영어로 새는 것을 막기 위해 언어를 한 번 더 못박습니다.
KO = "질문(query)과 답변(answer)은 반드시 한국어로 작성하세요."

candidates = [
    SingleHopSpecificQuerySynthesizer(llm=generator_llm, llm_context=KO),
    MultiHopAbstractQuerySynthesizer(llm=generator_llm, llm_context=KO),
    MultiHopSpecificQuerySynthesizer(llm=generator_llm, llm_context=KO),
]

synthesizers = []
for s in candidates:
    # 그래프에 이 질문 유형을 만들 재료(개체명/주제/관계)가 없으면 건너뜀
    # (관계가 아예 없으면 빈 리스트 대신 ValueError 가 나므로 둘 다 처리)
    try:
        usable = bool(s.get_node_clusters(kg))
    except ValueError:
        usable = False
    if not usable:
        print("재료 부족으로 제외:", s.name)
        continue
    # 프롬프트 예시를 한국어로 번역해서 적용
    prompts = await s.adapt_prompts("korean", llm=generator_llm)
    s.set_prompts(**prompts)
    synthesizers.append(s)

# 사용 가능한 유형끼리 같은 비율 (합계 1.0)
query_distribution = [(s, 1 / len(synthesizers)) for s in synthesizers]
[(s.name, round(w, 2)) for s, w in query_distribution]

재료 부족으로 제외: multi_hop_abstract_query_synthesizer


[('single_hop_specific_query_synthesizer', 0.5),
 ('multi_hop_specific_query_synthesizer', 0.5)]

In [29]:
testset = generator.generate(
    testset_size=4,
    query_distribution=query_distribution,
    run_config=run_config,
    raise_exceptions=False,  # 일부 질문 생성이 실패해도 나머지는 계속
)

test_df = testset.to_pandas()
test_df[["user_input", "reference", "synthesizer_name"]]

Generating Samples: 100%|██████████| 4/4 [00:02<00:00,  1.48it/s]


,user_input,reference,synthesizer_name
0,2023년 10월 30일에 발표된 바이든 대통령의 행정명령은 어떤 내용을 포함하고 ...,"2023년 10월 30일, 바이든 대통령은 안전하고 신뢰할 수 있는 AI 개발과 사...",single_hop_specific_query_synthesizer
1,바이든 대통려가 서명한 행정명령의 주요 내용은 무엇인가요?,바이든 대통령이 서명한 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령...,single_hop_specific_query_synthesizer
2,2023년 10월 30일에 발표된 미국의 AI 관련 행정명령과 연방거래위원회가 제출...,2023년 10월 30일에 발표된 미국의 AI 관련 행정명령은 AI의 안전과 보안 ...,multi_hop_specific_query_synthesizer
3,"프랑스는 AI 안전성 정상회의에서 어떤 역할을 하고 있으며, G7의 AI 국제 행동...","프랑스 정부는 AI 안전성 정상회의에서 1년 후 대면 정상회의를 개최할 예정이며, ...",multi_hop_specific_query_synthesizer


In [30]:
test_df.to_csv("../data/ragas_testset_ko.csv", index=False)

---
## 예제 3. 평가 대상 RAG 만들기

같은 5페이지로 간단한 RAG 를 만들고, 테스트 질문마다 **검색 문서(`retrieved_contexts`)** 와 **답변(`response`)** 을 수집합니다.

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

splits = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50).split_documents(docs)
vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings(model="text-embedding-3-small"))

rag_prompt = PromptTemplate.from_template(
    """문맥만 사용해서 한국어로 답하세요. 문맥에 없으면 모른다고 답하세요.

#문맥:
{context}

#질문:
{question}

#답변:"""
)
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def run_rag(questions, k=4):
    """질문 리스트 → (검색 문서 리스트, 답변 리스트)"""
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    contexts = [[d.page_content for d in ds] for ds in retriever.batch(questions)]
    answer_chain = rag_prompt | rag_llm | StrOutputParser()
    responses = answer_chain.batch(
        [{"context": "\n\n".join(c), "question": q} for c, q in zip(contexts, questions)]
    )
    return contexts, responses


questions = test_df["user_input"].tolist()
retrieved_contexts, responses = run_rag(questions, k=4)
print(responses[0])

2023년 10월 30일에 발표된 바이든 대통령의 행정명령은 다음과 같은 내용을 포함하고 있습니다: AI의 안전과 보안 기준 마련, 개인정보보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제협력.


---
## 예제 4. `metrics.collections` 로 채점 (0.4 권장 방식)

- LLM 은 `llm_factory()` 로 만든 **구조화 출력(Instructor)** LLM 을 사용합니다. (LangChain 모델이 아님)
- 각 지표는 **필요한 입력만 인자로** 받습니다. → 어떤 컬럼이 필요한지 코드만 봐도 알 수 있습니다.
- `abatch_score()` 로 여러 건을 한 번에 채점합니다.

In [32]:
from openai import AsyncOpenAI
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    AnswerRelevancy,
    ContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
)

client = AsyncOpenAI()
eval_llm = llm_factory("gpt-4o-mini", client=client)
eval_emb = RagasOpenAIEmbeddings(client=client, model="text-embedding-3-small")

faithfulness = Faithfulness(llm=eval_llm)                                  # 답변이 검색 문서에 근거하나
answer_relevancy = AnswerRelevancy(llm=eval_llm, embeddings=eval_emb, strictness=1)  # 답변이 질문에 맞나
context_precision = ContextPrecisionWithReference(llm=eval_llm)            # 쓸모 있는 문서가 위에 있나
context_recall = ContextRecall(llm=eval_llm)                               # 필요한 정보를 다 찾았나

In [33]:
import pandas as pd

rows = [
    {"user_input": q, "response": a, "retrieved_contexts": c, "reference": ref}
    for q, a, c, ref in zip(questions, responses, retrieved_contexts, test_df["reference"])
]


def pick(rows, keys):
    """지표마다 필요한 입력만 골라서 넘깁니다."""
    return [{k: r[k] for k in keys} for r in rows]


scores = {
    "faithfulness": await faithfulness.abatch_score(pick(rows, ["user_input", "response", "retrieved_contexts"])),
    "answer_relevancy": await answer_relevancy.abatch_score(pick(rows, ["user_input", "response"])),
    "context_precision": await context_precision.abatch_score(pick(rows, ["user_input", "retrieved_contexts", "reference"])),
    "context_recall": await context_recall.abatch_score(pick(rows, ["user_input", "retrieved_contexts", "reference"])),
}

score_df = pd.DataFrame({name: [r.value for r in results] for name, results in scores.items()})
score_df.insert(0, "user_input", questions)
score_df

,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,2023년 10월 30일에 발표된 바이든 대통령의 행정명령은 어떤 내용을 포함하고 ...,1.000000,0.387610,1.0,1.0
1,바이든 대통려가 서명한 행정명령의 주요 내용은 무엇인가요?,1.000000,0.225408,1.0,1.0
2,2023년 10월 30일에 발표된 미국의 AI 관련 행정명령과 연방거래위원회가 제출...,1.000000,0.561732,1.0,1.0
3,"프랑스는 AI 안전성 정상회의에서 어떤 역할을 하고 있으며, G7의 AI 국제 행동...",0.666667,0.666862,1.0,1.0


In [34]:
# 지표별 평균
score_df.drop(columns="user_input").mean().round(3)

faithfulness         0.917
answer_relevancy     0.460
context_precision    1.000
context_recall       1.000
dtype: float64

가장 점수가 낮은 문항을 골라 **어느 단계(검색/생성)의 문제인지** 확인합니다.

In [35]:
worst = score_df["faithfulness"].idxmin()
print("질문:", questions[worst])
print("답변:", responses[worst][:300])
print("faithfulness:", scores["faithfulness"][worst].value)

# 같은 문항의 다른 지표도 함께 보면 원인을 좁힐 수 있습니다.
#  - context_recall 도 낮다 → 검색 문제 (필요한 문서를 못 찾음)
#  - context_recall 은 높은데 faithfulness 만 낮다 → 생성 문제 (문서에 없는 내용을 지어냄)
score_df.loc[worst]

질문: 프랑스는 AI 안전성 정상회의에서 어떤 역할을 하고 있으며, G7의 AI 국제 행동강령과 어떻게 연결되나요?
답변: 프랑스는 AI 안전성 정상회의에서 28개국 중 하나로 참여하여 AI 안전 보장을 위한 협력 방안을 논의하고 있습니다. 또한, 프랑스 정부는 한국과 함께 1년 후 대면 정상회의를 개최할 예정입니다. G7의 AI 국제 행동강령은 첨단 AI 시스템의 위험 식별과 완화에 대한 자발적인 채택을 권고하는 내용을 담고 있으며, 이는 AI 안전성 정상회의에서 논의된 AI 시스템의 안전 평가와 관련된 책임과 연결될 수 있습니다. 두 이니셔티브 모두 AI의 안전성과 책임성을 강조하고 있습니다.
faithfulness: 0.6666666666666666


user_input           프랑스는 AI 안전성 정상회의에서 어떤 역할을 하고 있으며, G7의 AI 국제 행동...
faithfulness                                                  0.666667
answer_relevancy                                              0.666862
context_precision                                                  1.0
context_recall                                                     1.0
Name: 3, dtype: object

---
## 예제 5. 설정값 비교: retriever `k`

같은 질문에 대해 검색 문서 수 `k` 만 바꿔서 검색 지표를 비교합니다.

- `k` ↑ → 필요한 정보를 찾을 확률 ↑ (**ContextRecall ↑**), 대신 쓸모없는 문서도 섞임 (**ContextPrecision ↓** 가능)
- `k` ↓ → 정밀하지만 놓치기 쉬움

In [36]:
compare = []
for k in [1, 4]:
    ctxs, _ = run_rag(questions, k=k)
    inputs = [
        {"user_input": q, "retrieved_contexts": c, "reference": ref}
        for q, c, ref in zip(questions, ctxs, test_df["reference"])
    ]
    recall = await context_recall.abatch_score(inputs)
    precision = await context_precision.abatch_score(inputs)
    compare.append({
        "k": k,
        "context_recall": sum(r.value for r in recall) / len(recall),
        "context_precision": sum(r.value for r in precision) / len(precision),
    })

pd.DataFrame(compare).round(3)

,k,context_recall,context_precision
0,1,0.792,1.0
1,4,1.000,1.0


---
## 정리

| 하고 싶은 것 | 쓰는 API |
|---|---|
| 한 번에 테스트셋 생성 | `TestsetGenerator.from_langchain(...).generate_with_langchain_docs(docs, n)` |
| 추출 결과 재사용 | `KnowledgeGraph` + `apply_transforms` + `kg.save/load` + `generator.generate(n)` |
| 질문 유형·비율 조절 | `query_distribution=[(Synthesizer, 비율), ...]` |
| 사용자층 반영 | `persona_list=[Persona(...)]` |
| 한국어 질문 | `await synth.adapt_prompts("korean", llm)` → `synth.set_prompts(**prompts)` |
| 채점 (새 방식) | `llm_factory` + `metrics.collections.*` + `ascore` / `abatch_score` |
| 채점 (레거시) | `evaluate(dataset, metrics, llm=LangchainLLMWrapper(...))`, ch05_02 참고 |